# 单边通信基础概念

本节是 HIXL 开发的基础章节，学习 HIXL 首先需要了解单边通信的概念及其核心优势。本节将按照以下结构介绍单边通信相关内容：

- 通信术语与相关概念
- 双边通信的局限
- 单边通信原理
- 单边通信与双边通信对比


## 1. 通信术语与相关概念

为便于理解后续内容，学习本课程前请先了解如下术语、缩略语及相关概念。

| 概念 | 全称 | 解释说明 |
| --- | --- | --- |
| NPU | Neural Network Processing Unit | 神经网络处理单元，是昇腾 AI 计算任务的主要执行设备 |
| AI 节点 | AI Server | 昇腾 AI Server，通常是 8 卡或 16 卡的昇腾 NPU 设备组成的服务器形态的统称 |
| AI 集群 | AI POD | AI 集群由多个节点通过交换机（Switch）互联组成，用于分布式训练或推理系统 |
| Host |  | 指与 Device 相连接的 x86 或 arm 服务器，利用 Device 提供的计算能力完成业务 |
| Device |  | 指继承了计算芯片的硬件设备，如果 NPU/GPU 等，利用 PCIe 接口与 Host 侧连接，为 Host 提供高能性计算能力 |
| HIXL | Huawei Xfer Library | 华为昇腾单边通信库，面向集群场景提供简单、可靠、高效的点对点数据传输能力 |
| HCCL | Huawei Collective Communication Library | 华为集合通信库，提供单机多卡以及多机多卡间的数据并行、模型并行集合通信能力 |
| PCIe | Peripheral Component Interconnect Express | 一种串行外设扩展总线标准，常用于计算机系统中的外设扩展 |
| SDMA | System Direct Memory Access | 系统直接内存访问技术，简称 DMA，允许设备侧搬运内存数据，而不需要 CPU 的干预 |
| RDMA | Remote Direct Memory Access | 远程直接内存访问技术，能够直接将数据从一台机器的内存传输到另一台机器，无需双方操作系统介入，一般指可以跨网络的内存访问方式 |
| RoCE | RDMA over Converged Ethernet | 承载在融合以太网上的 RDMA 技术，即跨越以太网的 RDMA 通信方式。RoCE 有 v1 和 v2 两个版本，本课程中 RoCE 特指 RoCE v2 |
| HCCS | Huawei Cache Coherence System | 华为缓存一致性系统，用于 CPU/NPU 之间高速互联 |
| D2rH | Device to Remote Host | 指数据从 Device 设备往远端 Host 设备写数据，这里的远端区别于通过 PCIe 直通的 D2H 拷贝 |
| rH2D | Remote Host to Device | 指数据从 Device 设备往远端 Host 设备读数据，这里的远端区别于通过 PCIe 直通的 H2D 拷贝 |
| rD2H | remote Device to Host | 指数据从 Host 设备往远端 Device 设备读数据，这里的远端区别于通过 PCIe 直通的 D2H 拷贝 |
| H2rD | Host to remote Device | 指数据从 Host 设备往远端 Device 设备写数据，这里的远端区别于通过 PCIe 直通的 H2D 拷贝 |
| 通信域 |  | 集合通信执行上下文，管理通信实体（例如一个 NPU 就是一个通信实体）和资源 |
| Rank |  | 通信域中的通信实体标识，通常是 0 到 n-1（n 为通信域中通信实体的数量）的唯一编号 |


## 2. 双边通信的局限

传统网络收发采用「双边通信」：发送方调用 `send`，接收方调用 `recv`，双方均需准备好，协同参与数据的传输。类似于快递签收模式，发送和接收双方均需显式参与。

典型的双边通信模型：传统 Socket、MPI Send/Recv、RDMA Send/Recv。

双边通信数据传输流程如下：

<img src="./images/two_sided_comm_flow.png" alt="双边通信流程" width="40%">

- ① 发送请求：节点 A 向节点 B 发起通信请求
- ② 接收并处理请求：节点 B 的 CPU 参与请求处理
- ③ 返回响应：节点 B 确认准备好接收
- ④ 数据传输：节点 A 发送实际数据
- ⑤ 接收并存储数据：节点 B 的 CPU 参与数据搬运
- ⑥ 确认完成：节点 B 确认数据接收完毕

> 在数据传输过程中，接收方 CPU 需要多次参与交互，交互协议繁琐，CPU 占用率高。

双边通信数据完整传输路径如下：

<img src="./images/two_sided_data_path.png" alt="双边通信数据路径" />

- 数据从发送方应用出发，经过 CPU 拷贝进入发送缓冲区
- 通过网络传输到达接收方的接收缓冲区
- 再经过 CPU 拷贝最终到达接收方应用

> 双边通信数据传输链路冗长，且在传输过程中存在多次应用内存和缓冲区内存间的中间拷贝，存在额外的性能开销。

在 AI 大模型训练/推理场景下，数据张量、模型参数和 KV Cache 等大块数据/高频小块数据的传输对时延都有很高的要求，而双边通信存在 **CPU 控制开销大**、**数据路径冗长**、**多余的数据拷贝**等局限，会对数据传输性能造成很大的影响。

相比于双边通信，下面要介绍的“单边通信”技术能更好地匹配大模型训推对高性能数据传输的要求。


## 3. 单边通信原理

「单边通信」的含义：通信发起方可以直接读写远端接收方内存，远端 CPU 不参与数据传输过程。

**要特别说明的是：“远端 CPU 不参与数据传输”并不表示接收方全程什么都不需要做**，通常接收方需要提前完成以下准备动作：

- 准备接收内存：申请缓冲区，注册为可被通信设备访问的内存区域；
- 分享访问信息：将远端地址、长度、访问权限、密钥等信息告知发起端；
- 建立通信通道：双方初始化连接、创建必要的通信资源。
> 单边通信准备动作只用在初始化阶段做一次，后续每次数据传输无需重复以上动作，这是和双边通信的核心区别之一。

典型的单边通信模型：MPI RMA、RDMA PUT/GET 以及本课程的主角“HIXL”。

单边通信数据传输流程如下：

<img src="./images/one_sided_comm_flow.png" alt="单边通信流程" width="60%">

- ① 提交 RDMA 请求：节点 A 将数据传输请求提交给 RDMA 网卡
- ② 直接写入远程内存：通过 RDMA 网卡直接访问并写入节点 B 的内存，全程无需节点 B 的 CPU 参与
- ③ 完成通知：网卡向节点 A 返回操作完成状态

> 在数据传输过程中，接收方 CPU 无需参与，直接由 RDMA 网卡完成数据的搬运。

单边通信数据完整传输路径如下：

<img src="./images/one_sided_data_path.png" alt="单边通信数据路径" width="60%">

- 数据从发起方应用出发，通过 RDMA 网卡直接写入远程内存的目标地址
- 全程由网卡硬件完成，无需中间缓冲区，实现零拷贝传输

> 单边通信数据传输链路简单，无多余的中间拷贝开销。
>
> 注意：零拷贝（zero-copy）不是「数据完全不移动」，数据仍需从源地址移动到目标地址，只是传输过程没有中转导致的多余拷贝。

注意：**远端接收方无法感知数据发起方传输完成，** 发起方通常需要通过控制面消息通知接收方，该动作一般由上层应用框架完成。


## 4. 单边通信与双边通信对比

### 4.1 核心区别

单边通信和双边通信的核心区别总结如下：

| 特性 | 单边通信 | 双边通信 |
| --- | --- | --- |
| **发起方式** | 单侧发起 | 双边协商，需通信域 |
| **远端 CPU** | 零开销，无需介入 | 需远端响应和处理 |
| **通信时延** | 低（零拷贝、无 CPU 干预） | 高（多次拷贝、CPU 参与） |
| **可靠性控制** | 需要额外的控制机制保证时序 | 由协议层保证可靠传输 |
| **典型模型** | RDMA PUT/GET、Hixl | Socket、Send/Recv |

>

### 4.2 场景选择

单边通信的优势在于远端 CPU 不参与每次数据搬运关键路径上接收和拷贝，传输过程简单高效，因此更适合大块数据/高频小块数据搬运等数据面场景。

双边通信的优势在于通信模型更加直观可靠，同步控制机制更加简单清晰，传输数据大小灵活，因此更适合 RPC、请求-响应、消息通知等控制面场景。


## 课后练习
本节介绍了单边通信和双边通信的原理和区别，请根据学习内容完成以下题目进行自测。

1. （判断题）RDMA（远程直接内存访问）技术能够实现跨网络的内存访问，整个传输过程无需双方操作系统介入。

2. （判断题）零拷贝（zero-copy）表示数据完全不移动，从源地址到目标地址不需要任何传输过程。

3. （单选题）在单边通信中，远端接收方无法感知数据传输完成，发起方通常需要通过什么方式通知接收方？
    A. 发送完成中断
    B. 控制面消息通知
    C. 双边协商确认
    D. 接收方主动查询网络状态

4. （单选题）以下哪个场景不适合使用单边通信？
    A. 大块模型参数的跨节点传输
    B. 高频小块数据的张量同步
    C. RPC 远程过程调用和消息通知
    D. KV Cache 的跨节点迁移

5. （多选题）单边通信相比双边通信具有哪些核心区别？
    A. 发起方式为单侧发起，无需双边协商
    B. 远端 CPU 零开销，无需介入每次数据传输
    C. 通信时延低，实现零拷贝传输
    D. 可靠性传输由协议层自动保证

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/01.02_answer.txt